# <span style = "color:rebeccapurple"> Classification</span>

<span style="text-transform: uppercase;
        font-size: 14px;
        letter-spacing: 1px;
        font-family: 'Segoe UI', sans-serif;">
    Author
</span>

efrén cruz cortés

<hr style="border: none; height: 1px; background: linear-gradient(to right, transparent 0%, #ccc 10%, transparent 100%); margin-top: 10px;">

## <span style = "color:darkorange"> Concepts - classification, generalization

See slides

## <span style = "color:darkorchid"> Imports and Datasets

In [ ]:
# :: IMPORTS ::

# Scikit-learn specifics:
from sklearn import datasets
from sklearn import preprocessing
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Helper modules
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# :: DATA ::
wine_directory = "data/wine_cultivars.csv"


OK, it's time to get into machine learning models! Let's start with classification.

## <span style = "color:darkorchid"> Alice: sommelière extraordinaire!

After coming back from her cold polar adventures, Alice is ready for a change of scenery, so she takes a trip to Italy. Being quite the sommelière herself, she visits her favorite ristorante in La Toscana, *Il Cappellaio Matto*, in search of good wine. As it turns out, there is a heated debate going on, everyone is wondering if sommelieres can actually distinguish different wines, or if they are just faking it.

Alice decides to settle the dispute with machine learning. Her colleagues trust her, so they give her a dataset containing the chemical composition of different wine samples. All these samples were grown in the Toscana region, but they were made of three different cultivars (a cultivar is a specific plant variety, in this case varieties of grape vines).

![alice-sommerlier](images/alice_sommelier.png){width=30%}

 ## <span style = "color:darkorchid">Hands-on classification

 ### <span style="color:teal"> Load and inspect the data

In [ ]:
# Load data
wine = pd.read_csv(wine_directory)

In [ ]:
# Let'see what's inside
wine.head()

Our task is to distinguish cultivars, so the target is the *cultivars* column. Let's separate the target from the predictive features:

In [ ]:
wine_X = wine.drop('cultivar', axis='columns')
wine_y = wine['cultivar']

In [ ]:
print(f"X_dimensions = {wine_X.shape}")
wine_X.head()

In [ ]:
print(f"y labels = {wine_y.unique()}")
wine_y.head()

Quick tip: since we're using a jupyter notebook and cells run somehow independent. It's best practice to do any loading and splitting in a unified cell:

In [ ]:
wine = pd.read_csv(wine_directory)
wine_X = wine.drop('cultivar', axis='columns')
wine_y = wine['cultivar']

### <span style = "color:teal"> IMPORTANT - The train-test split!!

We mentioned in the slides that in order to make any statement about generalization, we must be able to test our classifier on **unseen** data. To do so, we split our whole dataset into *training* data, used to train our classifier, and *testing* data, used to test it. Under no circumstance can the classifier access the testing before the final test.

In [ ]:
# scikit-learn makes it easy for us to randomly split the data
wX_train, wX_test, wy_train, wy_test = train_test_split(wine_X, wine_y, test_size = .15)

### <span style = "color:teal"> Using a support vector classifier

For now, we will skip the preprocessing stage and head straight to the classification task. `scikit-learn` provides different classifiers for us, let's start with the support vector machine (SVM). You can see the documentation [here](https://scikit-learn.org/stable/modules/svm.html). We can find support vector classifiers in the `svm` module, which we have already imported.

In [ ]:
# Create SVM classifier:
wine_clf = svm.SVC()

In [ ]:
# let's see our classifier
wine_clf

In [ ]:
# Fit the SVM classifier:
wine_clf.fit(wX_train, wy_train)

In [ ]:
wine_clf.classes_

In [ ]:
# Predict the cultivars
wine_predictions = wine_clf.predict(wX_test)
wine_predictions

Let's compare the predictions to the true values:

In [ ]:
comparison_df = pd.DataFrame(data = {"Predicted": wine_predictions,
                                    "True Cultivars": wy_test})

In [ ]:
comparison_df.head(10)

Let's check now our accuracy (the proportion of times we got it right):

In [ ]:
# Evaluate:
wine_clf.score(wX_test, wy_test)

### <span style = "color:teal"> Adding Preprocessing

As stated before, the type of preprocessing we must do depends on your data and your model. All our features are numeric, which makes things easier. Let's go with the standard scaler for now.

In [ ]:
# Create and fit preprocessor
wine_preprocessor = preprocessing.StandardScaler().fit(wX_train, wy_train)

In [ ]:
# Transform data to standard scale
wX_train_trans = wine_preprocessor.transform(wX_train)

In [ ]:
# Fit classifier to transformed data
wine_clf = svm.SVC().fit(wX_train_trans, wy_train)

In [ ]:
# Evaluate on testing data:
    # Transform test data
wX_test_trans = wine_preprocessor.transform(wX_test)

    # Check accuracy
wine_clf.score(wX_test_trans, wy_test)

Woah!! Our accuracy skyrocketed, looks like preprocessing is very important uh?

### <span style = "color:teal"> Putting it all together

In [ ]:
# Step 1: Load the data
wine = pd.read_csv('data/wine_cultivars.csv')
wine_X = wine.drop("cultivar", axis="columns")
wine_y = wine["cultivar"]

# Step 2: Split the data
seed = 42
wX_train, wX_test, wy_train, wy_test = train_test_split(wine_X, wine_y, test_size = .3, random_state=seed)

# Step 3: Create preprocessor, fit it, transform the data
wine_preprocessor = preprocessing.StandardScaler().fit(wX_train, wy_train)
wX_train_trans = wine_preprocessor.transform(wX_train)

# Step 4: Create classifier, fit it
wine_clf = svm.SVC().fit(wX_train_trans, wy_train)


# Step 5: Evaluation
    # First make sure you also transform the test data
wX_test_trans = wine_preprocessor.transform(wX_test)

    # You can predict label for individual points:
print(wine_clf.predict(wX_test_trans[0:3]))     # <-- Predict class for first 3 test datapoints

    # And finally evaluate the accuracy of the classifier on the whole test dataset
wine_clf.score(wX_test_trans, wy_test)

### <span style = "color:teal"> Visualization

To build some intuition on classification, we'll need to make a 2D plot. Below I'm plotting two features of the wine dataset, together with the classification regions. The colored regions is what the SVM decides to classify as one class or another. Each point is colored by its true label. Note that we actually have 13 predictive features not just two, so we're seeing only a flat version of the data.

(The code for generating this image is found in `support_materials.ipynb`)

![classification-visualization](images/classification_viz_wine.png){width=50%}

## <span style = "color:darkorange"> Intermezzo - Pipelines

Notice it is a bit annoying to preprocess the training and testing data independently, and keep track of all steps of the pipeline. Indeed, as our machine learning project grows and becomes more robust, so does our pipeline (from the pre-processing stage, which by the way may require different methods for different columns, to evaluation). Lucky for us, `scikit-learn` has a `Pipeline` class that can help us contain all stages of our project.

The syntax is pretty simple. You create the objects independently and then put them together in the pipeline:

In [ ]:
wine_pipeline = Pipeline(
    [
        ("preprocessor", preprocessing.StandardScaler()),
        ("classifier", svm.SVC())
    ]
)

And we can look at our pipeline:

In [ ]:
wine_pipeline

Each stage of the pipeline keeps the same syntax, so we can use familiar methods like `.fit`, `.transform`, `.predict`, etc. Here's our previous example using `Pipeline()`.

Notice we don't need to do any transformation ourselves, even for the testing data!!

In [ ]:
# Step 1: Load the data
wine = pd.read_csv('data/wine_cultivars.csv')
wine_X = wine.drop("cultivar", axis="columns")
wine_y = wine["cultivar"]

# Step 2: Split the data
seed = 42
wX_train, wX_test, wy_train, wy_test = train_test_split(wine_X, wine_y, test_size = .3, random_state=seed)

# Step 3: Create the pipeline
wine_pipeline = Pipeline(
    [
        ("preprocessor", preprocessing.StandardScaler()),
        ("classifier", svm.SVC())
    ]
)

# Step 4: Fit the pipeline
wine_pipeline.fit(wX_train, wy_train)

# Step 5: You can evaluate the accuracy of the classifier on the whole test dataset
wine_pipeline.score(wX_test, wy_test)

You can learn more about pipelines in the `extra_materials` folder.

## <span style = "color:red">Astronomy Exercise

Having learned from Alice how to perform classification, Bo now has an idea. In their Italian trip Bo learned that the ancient Romans interpreted comets as omens, for example a disastrous event, or as a divine message, like [Caesar's Comet](https://en.wikipedia.org/wiki/Caesar%27s_Comet). Bo obtains a comet dataset, and decides to investigate.

Yesterday you looked at the NEOComets dataset. Here's a review:

**Near-Earth Object (NEO) Comets**: This dataset contains orbital and physical data for comets that have been classified as near-Earth objects. It was downloaded from the JPL Small-Body Database. Each row represents a comet (or a fragment of a comet). The columns include:

- **full_name**: The comet's official designation
- **class**: Orbital class — JFc (Jupiter-family comet), HTC (Halley-type comet), ETc (Encke-type comet), and others
- **diameter**: Nucleus diameter in kilometers
- **e**: Orbital eccentricity (0 = circular, closer to 1 = more elongated)
- **a**: Semi-major axis in AU (astronomical units; 1 AU = Earth–Sun distance)
- **i**: Orbital inclination in degrees (tilt relative to Earth's orbital plane)
- **per_y**: Orbital period in years

Create a classifier that predicts the Orbital class. Use the `Pipeline` object.

In [ ]:
# I will help you by selecting non-missing data and relevant columns only

# load the data
comets = pd.read_csv("data/NEOcomets.csv")

# specifyc features and target
comet_features = ['diameter', 'e', 'a', 'i', 'per_y']
comet_target = ['class']

# select relevant columns and drop missing data
comets = comets[comet_features + comet_target]
comets.dropna(inplace=True)

# preview
comets.head()

## <span style = "color:darkorchid"> Hyperparameters

Many machine learning algorithms depend on a few hyperparameters, which can be very influential for performance. Let's talk a bit about the hyperparameters for the support vector classifier.

**Regularization.** Regularization is the process of penalizing complicated decision boundaries. For the `SVC` class, this is done through the parameter `C`.

**Local Influence.** The extent to which individual points influence the classifier can be controlled with the parameter `gamma`. If gamma is too large, training data points will not interact much when making up the classifier, effectively creating pockets of one class. If on the other hand, gamma is too small, the influence of points spread throughout space, effectively creating a large region for one class.

Let's see how they affect performance to build an intuition. The default value of `C` is $1$. The default value of `gamma` is a heuristic based on the data's variance.

In [ ]:
# let's make a function that takes in a value of C, and an sklearn dataset, and outputs the classification accuracy
def full_classification(data, C_param, gamma_param = None):
    if gamma_param:
        params = {'C': C_param, 'gamma':gamma_param}
    else:
        params = {'C': C_param}
    seed = 42
    X = data.data
    y = data.target
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = .25, random_state=seed)
    classif_pipeline = Pipeline([
        ("preprocessor", preprocessing.StandardScaler()),
        ("classifier", svm.SVC(**params))
    ])
    classif_pipeline.fit(X_train, y_train)
    accuracy = classif_pipeline.score(X_test, y_test)
    return accuracy

In [ ]:
# Load the data
wine = datasets.load_wine(as_frame = True)

# Get accuracy for different levels of C
C_vals = [0.01, .001, .5, 1, 10, 100, 1000]
accs = []
for c in C_vals:
    accs.append(full_classification(wine, c))
print(accs)

Let's test `gamma` now:

In [ ]:
# Load the data
wine = datasets.load_wine(as_frame = True)

# Get accuracy for different levels of C
gamma_vals = [.001, .01, .1, .5, 1, 10, 100]
accs = []
for g in gamma_vals:
    accs.append(full_classification(wine, C_param=1, gamma_param=g))
print(accs)

We'll get back to hyperparameters in the next two sections, when we look at evaluation and cross-validation.

## <span style="color:darkorchid">Other Classifiers</span>

Finally, while we do not have time to go over other classification algorithms, `sklearn` offers a variety, including nearest neighbors and random forests. For more details refer to the [documentation](https://scikit-learn.org/stable/supervised_learning.html).

## <span style = "color:red"> Optional Exercise - Bo's daunting dilemma

Having followed Alice to Italy, Bo travels south to visit the famous city of Pompei. In there, they find the walls are full of ancient romans' graffiti! Bo would like to have a dataset of all the text in the walls, but copying it one by one would be incredibly daunting. They instead decide to take pictures and figure it out later.

Back in their lab, Bo needs to create a classifier that takes as inputs images of hand-written text, and maps them to their proper symbol. To begin, Bo will focus on numerical digits.

Load the "digits" dataset from `scikit-learn` and build a classifier to achieve this task.

![pompeii-graffiti](images/pompeii_02.png){width=40%}

Good luck!

**Notes**

- Each data point is an 8x8 pixels image
- An image is basically a matrix of numbers, where numbers at each position represent color values (grey scale)
- You can "flatten" this matrix into a long vector, which will be your feature vector
- The labels are the actual characters for the different numbers (0, 1, 2, etc.)
- Your dataset will contain 'images', which is a 3D array representing all images (N, 8, 8)
- Your dataset will contain 'data', which is the flattened array (N, 64)

In [ ]:
# We will load the "digits" dataset from scikit-learn, I'll do it for you:
digits = datasets.load_digits()

This `digits` object contains more than just the numerical data:

In [ ]:
# These are the pieces of information we have besides the data:
print(digits.keys(), '\n')

# For example, we can find a description of the dataset
print("DESCRIPTION\n", digits.DESCR)

In [ ]:
# Let me show you an image, in numeric matrix form
digits.images[0]

In [ ]:
# This is the flattened version, use it for classification
digits.data[0]

In [ ]:
# This is how the image actually looks, can you guess the digit?
plt.imshow(digits.images[42], cmap='Greys')
plt.show()

In [ ]:
# These are the true labels:
digits.target

In [ ]:
# Now build your own classification pipeline!
